In [26]:
# PRETRAINED CLOSED EXPERIMENT
import numpy as np
from scipy.stats import pearsonr
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
# List of your subjects
subject_ids = ['sub-01', 'sub-02', 'sub-03','sub-04','sub-06','sub-07','sub-08','sub-09','sub-10','sub-11','sub-13','sub-14','sub-15','sub-16','sub-17']
TR = 1.2 

In [29]:
def plot_eyesclosed_predictions(predictions, subject_ids=None, n_timepoints=313, cols=4, time_window=None, save_path=None):
    """
    Parameters:
    -----------
    time_window : tuple (start, end) or None
        Time window to plot, e.g. (50, 150). If None, plots from 0 to n_timepoints.
    save_path : str or None
        Directory to save the figure. If None, figure is not saved.
    """
    if subject_ids is None:
        subject_ids = sorted({k.split('/')[-1].split('_')[0] for k in predictions.keys()})

    # Resolve time window
    t_start = time_window[0] if time_window is not None else 0
    t_end   = time_window[1] if time_window is not None else n_timepoints

    subplot_titles = []
    valid_subjects = []
    for sub_id in subject_ids:
        pred_key = [k for k in predictions.keys() if sub_id in k]
        if not pred_key:
            print(f"No prediction found for {sub_id}")
            continue
        data = predictions[pred_key[0]]
        real_y = data['real_y'][t_start:t_end, 0, 0]
        pred_y = data['pred_y'][t_start:t_end, 0, 0]
        corr, _ = pearsonr(real_y, pred_y)
        r2 = r2_score(real_y, pred_y)
        subplot_titles.append(f"{sub_id}<br>r={corr:.2f}, R²={r2:.2f}")
        valid_subjects.append(sub_id)

    rows = int(np.ceil(len(valid_subjects) / cols))
    fig = make_subplots(rows=rows, cols=cols, subplot_titles=subplot_titles,
                        vertical_spacing=0.08, horizontal_spacing=0.05)

    for subplot_idx, sub_id in enumerate(valid_subjects):
        row = subplot_idx // cols + 1
        col = subplot_idx % cols + 1
        pred_key = [k for k in predictions.keys() if sub_id in k]
        data = predictions[pred_key[0]]
        real_y = data['real_y'][t_start:t_end, 0, 0]
        pred_y = data['pred_y'][t_start:t_end, 0, 0]
        x_axis = list(range(t_start, t_end))

        x_axis = [round((t - t_start) * 1.2, 2) for t in range(t_start, t_end)]

        fig.add_trace(go.Scatter(
            x=x_axis, y=real_y, mode='lines', line=dict(color='#0E1C36', width=2),
            name='real_y' if subplot_idx == 0 else None,
            showlegend=(subplot_idx == 0)),
            row=row, col=col
        )
        fig.add_trace(go.Scatter(
            x=x_axis, y=pred_y, mode='lines', line=dict(color='#5AC7DB', width=2),
            name='pred_y' if subplot_idx == 0 else None,
            showlegend=(subplot_idx == 0)),
            row=row, col=col
        )

        # Add alternating background squares (every 9 TRs = 10.8s)
        block_duration = 9 * TR  # 10.8 seconds
        n_blocks = 4

        for i in range(n_blocks):
            x0 = i * block_duration
            x1 = x0 + block_duration
            fig.add_vrect(
                x0=x0, x1=x1,
                fillcolor="lightgrey" if i % 2 == 0 else "white",
                opacity=0.3,
                layer="below",
                line_width=0,
            )

        # Vertical line every TR (1.2s)
        total_duration = (t_end - t_start) * TR
        for t in np.arange(0, total_duration + TR, TR):
            fig.add_vline(
                x=t,
                line=dict(color="grey", width=0.5, dash="dot"),
                layer="below",
            )

    fig.update_layout(
        width=1800,
        height=320 * rows,
        title=f"Real vs Predicted (Eyes Closed Model) per Subject — timepoints [{t_start}:{t_end}]",
        template="simple_white",
        showlegend=True
    )

    # --- Smart save ---
    if save_path is not None:
        subject_tag = valid_subjects[0] if len(valid_subjects) == 1 else "all_subjects"
        pred_name   = list(predictions.keys())[0].split('/')[-2] 
        window_tag  = f"_t{t_start}-{t_end}"
        filename    = f"{subject_tag}_{pred_name}{window_tag}.pdf"
        full_path   = os.path.join(save_path, filename)
        fig.write_image(full_path)
        print(f"Saved to: {full_path}")

    fig.show()


In [3]:
def adapt_evaluation(participant_evaluation):
    import pandas as pd
    import numpy as np
    pred_y = participant_evaluation["pred_y"]
    pred_y_median = np.nanmedian(pred_y, axis=1)
    pred_uncertainty = abs(participant_evaluation["euc_pred"])
    pred_uncertainty_median = np.nanmedian(pred_uncertainty, axis=1)
    df_pred_median = pd.DataFrame(
        np.concatenate(
            (pred_y_median, pred_uncertainty_median[..., np.newaxis]), axis=1),
        columns=["X", "Y", "Uncertainty"],
    )
    # With subTR
    subtr_values = np.concatenate((pred_y, pred_uncertainty[..., np.newaxis]),
                                  axis=2)
    index = pd.MultiIndex.from_product(
        [range(subtr_values.shape[0]),
         range(subtr_values.shape[1])],
        names=["TR", "subTR"])
    df_pred_subtr = pd.DataFrame(subtr_values.reshape(-1,
                                                      subtr_values.shape[-1]),
                                 index=index,
                                 columns=["X", "Y", "pred_error"])

    return df_pred_median, df_pred_subtr

In [4]:
# LOAD PRETRAINED DEEPMREYE 

evaluation_pretrained = np.load("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyelid_state/pred/evaluation_dict_eyelid_state_DeepMReye.npy", allow_pickle=True).item()
print(evaluation_pretrained['/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_eyelid_state/pp_data_base/sub-10_eyelid_state_labels.npz'].keys())

dict_keys(['real_y', 'pred_y', 'euc_pred', 'cv_fold'])


In [6]:
# LOAD CLOSED 

evaluation_closed = np.load("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyelid_state/pred/evaluation_dict_eyelid_state_DeepMReye_EyeStateTracking.npy", allow_pickle=True).item()
print(evaluation_closed['/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_eyelid_state/pp_data_eyestate/sub-10_eyelid_state_labels.npz'].keys())

dict_keys(['real_y', 'pred_y', 'euc_pred', 'cv_fold'])


In [8]:
# LOAD CALIB TRAINED 

evaluation_calibration = np.load("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyelid_state/pred/evaluation_dict_eyelid_state_DeepMReye_Calibration.npy", allow_pickle=True).item()
print(evaluation_calibration['/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_eyelid_state/pp_data_calib/sub-10_eyelid_state_labels.npz'].keys())

dict_keys(['real_y', 'pred_y', 'euc_pred', 'cv_fold'])


In [5]:
plot_eyesclosed_predictions(evaluation_pretrained, n_timepoints=313, cols=4, time_window= None)

In [7]:
plot_eyesclosed_predictions(evaluation_closed, n_timepoints=313, cols=4, time_window= None)

In [21]:
plot_eyesclosed_predictions(evaluation_closed, subject_ids= ['sub-04'], n_timepoints=313, cols=1, time_window= (118,154), save_path= "/Users/sinakling/Desktop/")


Saved to: /Users/sinakling/Desktop/sub-04_pp_data_eyestate_t118-154.pdf


In [30]:
plot_eyesclosed_predictions(evaluation_pretrained, subject_ids= ['sub-04'], n_timepoints=313, cols=1, time_window= (118,154), save_path= "/Users/sinakling/Desktop/")

Saved to: /Users/sinakling/Desktop/sub-04_pp_data_base_t118-154.pdf


In [ ]:
plot_eyesclosed_predictions(evaluation_calibration, subject_ids= ['sub-04'], n_timepoints=313, cols=1, time_window= (118,154), save_path= "/Users/sinakling/Desktop/")

Saved to: /Users/sinakling/Desktop/sub-04_pp_data_calib_t118-121.pdf


In [9]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import os

task_TRs = {
    "task_1": (5, 77),
    "task_2": (82, 154),
    "task_3": (159, 231),
    "task_4": (236, 308),
}

from scipy.stats import pearsonr

def run_subjectwise_taskwise_analysis_df(evaluation_dict, model, thresholds=[0.3], subject_ids=None):
    all_results = []
    conf_matrices = {}
    
    if subject_ids is None:
        subject_ids = list({k.split('_eyelid')[0] for k in evaluation_dict.keys()})
    
    for sub_id in subject_ids:
        pred_key = [k for k in evaluation_dict if sub_id in k]
        if not pred_key:
            print(f"No prediction found for {sub_id}")
            continue
        
        real_y = evaluation_dict[pred_key[0]]['real_y']
        pred_y = evaluation_dict[pred_key[0]]['pred_y']
        
        real_closure = real_y[:, 0, 0] if len(real_y.shape) == 3 else real_y.flatten()
        pred_closure = np.mean(pred_y[..., 0], axis=1) if len(pred_y.shape) == 3 else pred_y.flatten()
    
        
        min_len = min(len(real_closure), len(pred_closure))
        real_closure = real_closure[:min_len]
        pred_closure = pred_closure[:min_len]
        
        valid_idx = ~(np.isnan(real_closure) | np.isnan(pred_closure))
        real_closure = real_closure[valid_idx]
        pred_closure = pred_closure[valid_idx]
        
        if len(real_closure) == 0:
            print(f"No valid data for {sub_id}")
            continue
        
        for threshold in thresholds:
            real_binary = (real_closure >= threshold).astype(int)
            pred_binary = (pred_closure >= threshold).astype(int)
            
            for task, (start, end) in task_TRs.items():
                run_length = 313
                n_runs = len(real_binary) // run_length
                
                task_real = []
                task_pred = []
                run_corrs = []
                
                for r in range(n_runs):
                    offset = r * run_length
                    r_data = real_binary[offset + start: offset + end + 1]
                    p_data = pred_binary[offset + start: offset + end + 1]
                    task_real.append(r_data)
                    task_pred.append(p_data)
                    
                    # Continuous correlation
                    r_cont = real_closure[offset + start: offset + end + 1]
                    p_cont = pred_closure[offset + start: offset + end + 1]
                    if len(r_cont) > 1:
                        corr, _ = pearsonr(r_cont, p_cont)
                        run_corrs.append(corr)
                
                task_real = np.concatenate(task_real)
                task_pred = np.concatenate(task_pred)
                
                acc = accuracy_score(task_real, task_pred)
                precision = precision_score(task_real, task_pred, zero_division=0)
                hit_rate = recall_score(task_real, task_pred, zero_division=0)
                f1 = f1_score(task_real, task_pred, zero_division=0)
                
                cm = confusion_matrix(task_real, task_pred)
                conf_matrices[(sub_id, task, threshold)] = cm
                
                majority_class = int(np.mean(task_real >= 0.5) >= 0.5)
                baseline_acc = np.mean(task_real == majority_class)
                
                # Average correlation across runs
                mean_corr = np.mean(run_corrs) if run_corrs else np.nan
                
                all_results.append({
                    'subject': sub_id,
                    'model': model,
                    'task': task,
                    'threshold': threshold,
                    'accuracy': acc,
                    'hit_rate': hit_rate,
                    'precision': precision,
                    'f1_score': f1,
                    'baseline_accuracy': baseline_acc,
                    'corr': mean_corr  
                })
    
    results_df = pd.DataFrame(all_results)
    return results_df, conf_matrices


In [10]:
task_results_df_pretrained, task_conf_matrices_pretrained  = run_subjectwise_taskwise_analysis_df(evaluation_pretrained, model = "pretrained", thresholds=[0.1], subject_ids=subject_ids)

No prediction found for sub-01


/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A

In [11]:
task_results_df_closed, task_conf_matrices_closed  = run_subjectwise_taskwise_analysis_df(evaluation_closed,  model = "closed", thresholds=[0.1], subject_ids=subject_ids)

No prediction found for sub-01


/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A

In [12]:
task_results_df_calib, task_conf_matrices_calib  = run_subjectwise_taskwise_analysis_df(evaluation_calibration,  model = "calib", thresholds=[0.1], subject_ids=subject_ids)

No prediction found for sub-01


/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.

/Users/sinakling/softwares/anaconda3/envs/skling/lib/python3.11/site-packages/sklearn/metrics/_classification.py:386: UserWarning:

A

In [13]:
df_accuracies = pd.DataFrame()
df_accuracies = pd.concat([task_results_df_closed,task_results_df_pretrained, task_results_df_calib], ignore_index=True)
print(df_accuracies.groupby(['model', 'task'])['accuracy']
    .mean())

print(df_accuracies.groupby(['model', 'task'])['f1_score']
    .mean())

print(df_accuracies.groupby(['model', 'task'])['hit_rate']
    .mean())

model       task  
calib       task_1    0.566210
            task_2    0.788650
            task_3    0.571755
            task_4    0.999022
closed      task_1    0.553164
            task_2    0.797782
            task_3    0.581213
            task_4    0.999022
pretrained  task_1    0.606327
            task_2    0.762231
            task_3    0.613503
            task_4    0.999348
Name: accuracy, dtype: float64
model       task  
calib       task_1    0.354099
            task_2    0.869606
            task_3    0.385448
            task_4    0.999507
closed      task_1    0.348533
            task_2    0.873955
            task_3    0.387917
            task_4    0.999509
pretrained  task_1    0.350568
            task_2    0.848186
            task_3    0.363970
            task_4    0.999672
Name: f1_score, dtype: float64
model       task  
calib       task_1    0.596422
            task_2    0.891897
            task_3    0.571664
            task_4    0.999022
closed      t

In [14]:
print(df_accuracies.groupby(['model', 'task'])['corr']
    .mean())

model       task  
calib       task_1    0.198291
            task_2    0.869176
            task_3    0.199485
            task_4    0.831466
closed      task_1    0.183571
            task_2    0.872300
            task_3    0.202472
            task_4    0.837345
pretrained  task_1    0.227398
            task_2    0.857196
            task_3    0.186069
            task_4    0.741935
Name: corr, dtype: float64


In [15]:
from scipy.stats import permutation_test
import pandas as pd

def paired_permutation_tests(df, model_a, model_b, value):
    """
    Run paired permutation tests between two models for each task.

    Parameters:
        df (pd.DataFrame): DataFrame with columns ['subject', 'task', 'model', 'accuracy', 'corr']
        model_a (str): Name of the first model
        model_b (str): Name of the second model

    Returns:
        pd.DataFrame: A DataFrame with task, p-value, mean difference, and n_subjects
    """
    results = []

    grouped = df.groupby(['task'])
    for task, group in grouped:
        # Average across possible duplicates (subject, model)
        group_mean = group.groupby(['subject', 'model'])[value].mean().reset_index()

        # Pivot to get one row per subject, one column per model
        pivoted = group_mean[group_mean['model'].isin([model_a, model_b])] \
            .pivot(index='subject', columns='model', values=value) \
            .dropna()

        if pivoted.shape[0] < 2:
            continue  # skip if not enough subjects

        a_values = pivoted[model_a].values
        b_values = pivoted[model_b].values

        # Paired permutation test using mean difference as statistic
        result = permutation_test(
            (a_values, b_values),
            statistic=lambda x, y: (x - y).mean(),
            permutation_type='samples',
            vectorized=False,
            alternative='less',
            n_resamples=10000
        )

        results.append({
            'task': task,
            'model_a': model_a,
            'model_b': model_b,
            'mean_diff': (a_values - b_values).mean(),
            'p_value': result.pvalue,
            'n_subjects': len(pivoted)
        })

    return pd.DataFrame(results)


In [16]:
# Run paired permutation tests 
sig_df_pt_vs_closed_acc = paired_permutation_tests(df_accuracies, model_a='pretrained', model_b='closed', value = 'accuracy')
sig_df_pt_vs_closed_corr = paired_permutation_tests(df_accuracies, model_a='pretrained', model_b='closed', value = 'corr')

# Run paired permutation tests 
sig_df_calib_vs_closed_acc = paired_permutation_tests(df_accuracies, model_a='calib', model_b='closed', value = 'accuracy')
sig_df_calib_vs_closed_corr = paired_permutation_tests(df_accuracies, model_a='calib', model_b='closed', value = 'corr')

sig_df_pt_vs_calib_acc = paired_permutation_tests(df_accuracies, model_a='pretrained', model_b='calib', value = 'accuracy')
sig_df_pt_vs_calib_corr = paired_permutation_tests(df_accuracies, model_a='pretrained', model_b='calib', value = 'corr')

# Append the results from both pairings
sig_df_acc = pd.concat([sig_df_pt_vs_closed_acc, sig_df_calib_vs_closed_acc, sig_df_pt_vs_calib_acc], ignore_index=True)

sig_df_corr = pd.concat([sig_df_pt_vs_closed_corr, sig_df_calib_vs_closed_corr,sig_df_pt_vs_calib_corr], ignore_index=True)

# Show the combined dataframe with the significance results
#print(sig_df)
print(sig_df_acc)
print(sig_df_corr)

         task     model_a model_b  mean_diff   p_value  n_subjects
0   (task_1,)  pretrained  closed   0.053164  0.829917          14
1   (task_2,)  pretrained  closed  -0.035551  0.011099          14
2   (task_3,)  pretrained  closed   0.032290  0.850515          14
3   (task_4,)  pretrained  closed   0.000326  1.000000          14
4   (task_1,)       calib  closed   0.013046  0.617438          14
5   (task_2,)       calib  closed  -0.009132  0.270073          14
6   (task_3,)       calib  closed  -0.009459  0.407459          14
7   (task_4,)       calib  closed   0.000000  0.745825          14
8   (task_1,)  pretrained   calib   0.040117  0.896410          14
9   (task_2,)  pretrained   calib  -0.026419  0.029997          14
10  (task_3,)  pretrained   calib   0.041748  0.866013          14
11  (task_4,)  pretrained   calib   0.000326  1.000000          14
         task     model_a model_b  mean_diff   p_value  n_subjects
0   (task_1,)  pretrained  closed   0.043826  0.941906        

In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

model_order = ["pretrained","calib", "closed"]
x_positions = [0, 1, 2]
bar_width = 0.35
bar_color = 'rgba(66, 129, 164, 0.5)'

task_titles = {
    "task_1": "Eyes Open + Vision",
    "task_2": "Eyes Blink + Vision",
    "task_3": "Eyes Open + No Vision",
    "task_4": "Eyes Closed + No Vision"
}

model_name_map = {
    "pretrained": "DeepMReye",
    "closed": "DeepMReye + Closed", 
    "calib": "DeepMRye + Calib"
}

task_list = list(task_titles.keys())
n_tasks = len(task_list)

fig = make_subplots(
    rows=1, cols=n_tasks,
    subplot_titles=[task_titles[t] for t in task_list],
    horizontal_spacing=0.08
)

for col, task in enumerate(task_list, start=1):

    subset = df_accuracies[df_accuracies["task"] == task]

    # Pivot per subject for lines
    df_pivot = subset.pivot_table(
        index="subject", columns="model", values="accuracy"
    )[model_order].dropna() * 100

    means = df_pivot.mean()
    stderrs = df_pivot.sem()

    # Bars
    fig.add_trace(go.Bar(
        x=x_positions,
        y=means[model_order].values,
        error_y=dict(type='data', array=stderrs[model_order].values),
        text=[f"{v:.1f}" for v in means[model_order].values],
        textposition="inside",
        insidetextanchor="start",
        marker_color=bar_color,
        marker_line=dict(color=bar_color.replace('0.5', '0.9'), width=1),
        width=bar_width,
        showlegend=False,
    ), row=1, col=col)

    # Subject lines — grey, transparent, bigger dots
    for subject, row_data in df_pivot.iterrows():
        fig.add_trace(go.Scatter(
            x=x_positions,
            y=[row_data[m] for m in model_order],
            mode='lines+markers',
            line=dict(color='rgba(100, 100, 100, 0.35)', width=1.5),
            marker=dict(size=8, color='rgba(100, 100, 100, 0.5)'),
            showlegend=False,
        ), row=1, col=col)

    # Significance markers
    sig_df_acc["task"] = sig_df_acc["task"].apply(lambda x: x[0] if isinstance(x, tuple) else x)
    pair_count = 0
    for m1_i in range(len(model_order)):
        for m2_i in range(m1_i + 1, len(model_order)):
            model_a = model_order[m1_i]
            model_b = model_order[m2_i]

            sig_row = sig_df_acc[
                (sig_df_acc.task == task) &
                (
                    ((sig_df_acc.model_a == model_a) & (sig_df_acc.model_b == model_b)) |
                    ((sig_df_acc.model_a == model_b) & (sig_df_acc.model_b == model_a))
                )
            ]

            if not sig_row.empty and sig_row.iloc[0].p_value < 0.05:
                max_y = means[model_order].max()
                line_y = max_y + 5 + pair_count * 6

                fig.add_shape(
                    type="line",
                    x0=x_positions[m1_i],
                    x1=x_positions[m2_i],
                    y0=line_y, y1=line_y,
                    line=dict(color="black", width=1.5),
                    row=1, col=col
                )
                fig.add_trace(go.Scatter(
                    x=[(x_positions[m1_i] + x_positions[m2_i]) / 2],
                    y=[line_y + 2],
                    text=["*"],
                    mode="text",
                    showlegend=False
                ), row=1, col=col)

                pair_count += 1

    # Axes
    fig.update_yaxes(
        range=[0, 105],
        title_text="Classification Accuracy (%)" if col == 1 else "",
        row=1, col=col
    )
    fig.update_xaxes(
        tickvals=x_positions,
        ticktext=[model_name_map[m] for m in model_order],
        tickangle=15,
        row=1, col=col
    )

    # Chance line (25% for 4-class)
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(model_order) - 0.5,
        y0=50, y1=50,
        line=dict(color="black", width=1, dash="dot"),
        row=1, col=col
    )

fig.update_layout(
    height=550,
    width=300 * n_tasks + 100,
    title_text="All-Model Comparison Across Tasks",
    template="simple_white",
    showlegend=False,
    font=dict(family="Helvetica Neue", size=14, color="black"),
)

fig.show()
#fig.write_image("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyelid_state/figures/group/accuracy_plots_individualpoints.pdf")

In [18]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

model_order = ["pretrained", "calib", "closed"]
x_positions = [0, 1, 2]
bar_width = 0.35
bar_color = 'rgba(66, 129, 164, 0.5)'

task_titles = {
    "task_1": "Eyes Open + Vision",
    "task_2": "Eyes Blink + Vision",
    "task_3": "Eyes Open + No Vision",
    "task_4": "Eyes Closed + No Vision"
}

model_name_map = {
    "pretrained": "DeepMReye",
    "closed": "DeepMReye + Closed", 
    "calib": "DeepMreye + Calib"
}

task_list = list(task_titles.keys())
n_tasks = len(task_list)

fig = make_subplots(
    rows=1, cols=n_tasks,
    subplot_titles=[task_titles[t] for t in task_list],
    horizontal_spacing=0.08
)

for col, task in enumerate(task_list, start=1):

    subset = df_accuracies[df_accuracies["task"] == task]

    df_pivot = subset.pivot_table(
        index="subject", columns="model", values="corr"
    )[model_order].dropna()

    means = df_pivot.mean()
    stderrs = df_pivot.sem()

    # Bars
    fig.add_trace(go.Bar(
        x=x_positions,
        y=means[model_order].values,
        error_y=dict(type='data', array=stderrs[model_order].values),
        text=[f"{v:.2f}" for v in means[model_order].values],
        textposition="inside",
        insidetextanchor="start",
        marker_color=bar_color,
        marker_line=dict(color=bar_color.replace('0.5', '0.9'), width=1),
        width=bar_width,
        showlegend=False,
    ), row=1, col=col)

    # Subject lines
    for subject, row_data in df_pivot.iterrows():
        fig.add_trace(go.Scatter(
            x=x_positions,
            y=[row_data[m] for m in model_order],
            mode='lines+markers',
            line=dict(color='rgba(100, 100, 100, 0.35)', width=1.5),
            marker=dict(size=8, color='rgba(100, 100, 100, 0.5)'),
            showlegend=False,
        ), row=1, col=col)

    # Significance markers
    sig_df_corr["task"] = sig_df_corr["task"].apply(lambda x: x[0] if isinstance(x, tuple) else x)
    pair_count = 0
    for m1_i in range(len(model_order)):
        for m2_i in range(m1_i + 1, len(model_order)):
            model_a = model_order[m1_i]
            model_b = model_order[m2_i]

            sig_row = sig_df_corr[
                (sig_df_corr.task == task) &
                (
                    ((sig_df_corr.model_a == model_a) & (sig_df_corr.model_b == model_b)) |
                    ((sig_df_corr.model_a == model_b) & (sig_df_corr.model_b == model_a))
                )
            ]

            if not sig_row.empty and sig_row.iloc[0].p_value < 0.05:
                max_y = means[model_order].max()
                line_y = max_y + 0.05 + pair_count * 0.06  # scaled for 0-1 range

                fig.add_shape(
                    type="line",
                    x0=x_positions[m1_i],
                    x1=x_positions[m2_i],
                    y0=line_y, y1=line_y,
                    line=dict(color="black", width=1.5),
                    row=1, col=col
                )
                fig.add_trace(go.Scatter(
                    x=[(x_positions[m1_i] + x_positions[m2_i]) / 2],
                    y=[line_y + 0.02],
                    text=["*"],
                    mode="text",
                    showlegend=False
                ), row=1, col=col)

                pair_count += 1

    fig.update_yaxes(
        range=[-0.1, 1.15],
        title_text="Pearson Correlation" if col == 1 else "",
        row=1, col=col
    )
    fig.update_xaxes(
        tickvals=x_positions,
        ticktext=[model_name_map[m] for m in model_order],
        tickangle=15,
        row=1, col=col
    )

    # Chance line at r=0
    fig.add_shape(
        type="line",
        x0=-0.5, x1=len(model_order) - 0.5,
        y0=0, y1=0,
        line=dict(color="black", width=1, dash="dot"),
        row=1, col=col
    )

fig.update_layout(
    height=550,
    width=300 * n_tasks + 100,
    title_text="Model Comparison — Pearson Correlation Across Tasks",
    template="simple_white",
    showlegend=False,
    font=dict(family="Helvetica Neue", size=14, color="black"),
)

fig.show()